In [0]:
from pyspark.sql import functions as F

silver = "workspace.silver.adidas_us_sales"
dimension_gold = "workspace.gold.dim_canal_venta"
checkpoint = "/Volumes/workspace/gold/dim_canal_venta/checkpoint"


In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.gold.dim_canal_venta")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {dimension_gold} (
        id_canal_venta STRING,
        nombre_canal_venta STRING
    ) USING DELTA
""")

In [0]:
def cargar_dimension(lote, _):
    if lote.isEmpty():
        return

    dimension = lote.select(
        F.sha2("canal_venta", 256).alias("id_canal_venta"),
        F.col("canal_venta").alias("nombre_canal_venta"),
    ).distinct()
    dimension.createOrReplaceTempView("miembros_dimension")
    lote.sparkSession.sql(f"""
        MERGE INTO {dimension_gold} AS destino
        USING miembros_dimension AS origen
        ON destino.id_canal_venta = origen.id_canal_venta
        WHEN NOT MATCHED THEN INSERT *
    """)


In [0]:
flujo = (
    spark.readStream.option("readChangeFeed", "true").table(silver)
    .filter("_change_type IN ('insert', 'update_postimage')")
    .writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .foreachBatch(cargar_dimension)
    .start()
)
flujo.awaitTermination()
